In [1]:
# 1. Write your custom function here!

# For this demo, we do a basic NDVI calculation, inputting and returning a pandas DataFrame.
# Note that the function must input and return a pandas DataFrame. This is a requirement.
def compute_ndvi(df):
    # Perform your calculations
    df['ndvi'] = (df['band_2'] - df['band_1']) / (df['band_2'] + df['band_1'])
    return df["ndvi"]

In [2]:
def get_bands(df):
    return df[["SR_B4", "SR_B5"]]

In [ ]:
# 2. Authenticate to Google Earth Engine, if you haven't done so already on your machine!
import ee

ee.Authenticate()

In [3]:
# 3. Run some Earth Engine code! This will do a basic grab of some Landsat data, with some standard filtering. 
import ee
ee.Initialize(opt_url='https://earthengine-highvolume.googleapis.com')

# Basic cloud masking algorithm
def prep_sr_l8(image):
    # Bit 0 - Fill
    # Bit 1 - Dilated Cloud
    # Bit 2 - Cirrus
    # Bit 3 - Cloud
    # Bit 4 - Cloud Shadow
    qa_mask = image.select('QA_PIXEL').bitwiseAnd(int('11111', 2)).eq(0)
    saturation_mask = image.select('QA_RADSAT').eq(0)

    # Apply the scaling factors to the appropriate bands.
    optical_bands = image.select('SR_B.*').multiply(0.0000275).add(-0.2)
    thermal_bands = image.select('ST_B.*').multiply(0.00341802).add(149.0)

    # Replace the original bands with the scaled ones and apply the masks.
    return (image.addBands(optical_bands, None, True)
                 .addBands(thermal_bands, None, True)
                 .updateMask(qa_mask)
                 .updateMask(saturation_mask))

Plumas_Boundaries = ee.FeatureCollection("projects/robust-raster/assets/boundaries/Plumas_National_forest")
image = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterDate('2018-06-01', '2018-09-30').map(prep_sr_l8).select(['SR_B4', 'SR_B5']).median().set({"system:time_start": ee.Date("2018-06-01").millis()})
ic = ee.ImageCollection([image])

c:\anaconda3\envs\rr042_rpy2\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [ ]:
# 6. Do the full computation here!
# Click on the link to the Dask dashboard to view the computation in real time!
from robustraster import run

run(
    dataset=ic,
    source="ee",
    dataset_config={
        'geometry': Plumas_Boundaries,
        'crs': 'EPSG:3310',
        'scale': 30,
    },
    user_function_config={
        "user_function": get_bands,
        "user_function_args": (),
        "user_function_kwargs": {},
    },
    export_config={
        "mode": "raster",
        "file_format": "GTiff",
        "output_folder": "Plumas_Tiles",
        "vrt": True
    }
)

In [ ]:
from robustraster import run

run(
    dataset=r"./Plumas_Tiles/time_20180601T000000.vrt",
    source="local",
    user_function_config={
        "user_function": compute_ndvi,
        "user_function_args": (),
        "user_function_kwargs": {},
    },
    export_config={
        "mode": "raster",
        "file_format": "GTiff",
        "output_folder": "Plumas_NDVI_Tiles",
    },
    dask_mode="custom",
    dask_config={
        "n_workers": 6,
        "threads_per_worker": 1,
        "memory_limit": "3g",
    },
)